In [1]:
# 1. Kiwi 형태소 분석기 설치 (코랩은 실행 시마다 새로 설치해야 함)
!pip install kiwipiepy -q

# 2. 필요한 라이브러리 전체 임포트
import pandas as pd
import numpy as np
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import joblib
import os
from datetime import datetime
from google.colab import drive # Google Drive 연동을 위한 라이브러리

# 3. Google Drive 마운트
# 이 셀을 실행하면 인증 창이 뜰 거야. 안내에 따라 계정을 연결해 줘.
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 45.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 66.3 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# --- 중요: 자신의 Google Drive 경로에 맞게 수정 ---

# 1. 데이터셋 파일이 있는 경로
# 예: 내 드라이브(MyDrive)의 'project/dataset' 폴더 안에 파일이 있는 경우
DATASET_PATH = "/content/drive/MyDrive/master_dataset_final.csv" # <-- 이 부분을 수정해!

# 2. 학습된 모델을 저장할 폴더 경로
MODEL_SAVE_DIR = "/content/drive/MyDrive/1차모델저장" # <-- 이 부분을 수정해!

# ------------------------------------------------

# 모델 저장 폴더가 없으면 자동으로 생성
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

print(f"데이터셋 경로: {DATASET_PATH}")
print(f"모델 저장 경로: {MODEL_SAVE_DIR}")

데이터셋 경로: /content/drive/MyDrive/master_dataset_final.csv
모델 저장 경로: /content/drive/MyDrive/1차모델저장


In [3]:
# --- 프로세스 시작 ---
print("전체 데이터셋 랜덤 포레스트 모델 학습을 시작합니다...")
print(f"시작 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 1. 데이터 로드
print("\n1단계: 데이터 로드 및 확인")
df = pd.read_csv(DATASET_PATH)
print(f"- 로드된 데이터 크기: {df.shape}")
print(f"- 클래스 분포: {dict(df['is_phishing'].value_counts())}")

# 2. 키위 형태소 분석기 함수 정의
kiwi = Kiwi()
def tokenize_and_filter(text):
    result = kiwi.analyze(text)[0][0]
    tokens = []
    for word, pos, _, _ in result:
        if pos in {"NNG", "NNP", "VV", "VA"}:
            if pos in {"VV", "VA"}:
                word = word + "다"
            tokens.append(word)
    return " ".join(tokens)

# 3. 텍스트 데이터에 토크나이저 적용
print("\n2단계: 키위 토크나이저로 텍스트 전처리 중...")
df["processed_text"] = df["text"].astype(str).apply(tokenize_and_filter)
X = df["processed_text"]
y = df["is_phishing"]
print("전처리 완료")

전체 데이터셋 랜덤 포레스트 모델 학습을 시작합니다...
시작 시간: 2025-07-22 04:20:20

1단계: 데이터 로드 및 확인
- 로드된 데이터 크기: (12000, 4)
- 클래스 분포: {0: np.int64(6000), 1: np.int64(6000)}

2단계: 키위 토크나이저로 텍스트 전처리 중...
전처리 완료


In [4]:
# 3단계: 랜덤 포레스트 파이프라인 및 그리드 서치 구성
print("\n3단계: 파이프라인 및 그리드 서치 구성")
pipeline_rf = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True)),
    ("clf", RandomForestClassifier(random_state=42, class_weight="balanced")),
])

param_grid = {
    'tfidf__max_df': [0.9, 0.95],
    'tfidf__min_df': [5, 8],
    'tfidf__ngram_range': [(1, 2)],
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [None, 30, 50],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    pipeline_rf,
    param_grid,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=2
)

# 4단계: 그리드 서치 기반 모델 학습 시작
print("\n4단계: 그리드 서치 기반 모델 학습 (시간이 다소 소요됩니다)")
grid_search.fit(X, y)


3단계: 파이프라인 및 그리드 서치 구성

4단계: 그리드 서치 기반 모델 학습 (시간이 다소 소요됩니다)
Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('tfidf',
                                        TfidfVectorizer(sublinear_tf=True)),
                                       ('clf',
                                        RandomForestClassifier(class_weight='balanced',
                                                               random_state=42))]),
             n_jobs=-1,
             param_grid={'clf__max_depth': [None, 30, 50],
                         'clf__n_estimators': [100, 200],
                         'tfidf__max_df': [0.9, 0.95], 'tfidf__min_df': [5, 8],
                         'tfidf__ngram_range': [(1, 2)]},
             scoring='f1_weighted', verbose=2)

In [5]:
# 5단계: 최적화 결과 확인
print("\n5단계: 최적화 결과")
print("=" * 50)
print(f"최적 파라미터: {grid_search.best_params_}")
print(f"최적 F1 스코어 (CV): {grid_search.best_score_:.4f}")
print("=" * 50)

# 6단계: K-Fold 교차 검증 기반 일반화 성능 평가
print("\n6단계: K-Fold 교차 검증 기반 일반화 성능")
y_pred_cv = cross_val_predict(grid_search.best_estimator_, X, y, cv=cv)
print("교차 검증 성능 리포트:")
print(classification_report(y, y_pred_cv, target_names=['정상대화', '보이스피싱']))

# 7단계: 최적화된 모델 저장
print("\n7단계: 최적화된 랜덤 포레스트 모델 저장")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_filename = f"optimized_randomforest_model_{timestamp}.pkl"
model_path = os.path.join(MODEL_SAVE_DIR, model_filename)

joblib.dump(grid_search.best_estimator_, model_path)

# --- 최종 요약 ---
print("\n" + "=" * 60)
print("개선된 랜덤 포레스트 모델 학습 및 저장 완료!")
print("=" * 60)
print(f"완료 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"최종 F1 스코어 (CV): {grid_search.best_score_:.4f}")
print(f"저장된 최적 모델 경로: {model_path}")
print(f"저장된 모델 크기: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
print("=" * 60)


5단계: 최적화 결과
최적 파라미터: {'clf__max_depth': None, 'clf__n_estimators': 100, 'tfidf__max_df': 0.9, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2)}
최적 F1 스코어 (CV): 0.9957

6단계: K-Fold 교차 검증 기반 일반화 성능
교차 검증 성능 리포트:
              precision    recall  f1-score   support

        정상대화       1.00      0.99      1.00      6000
       보이스피싱       1.00      1.00      1.00      6000

    accuracy                           1.00     12000
   macro avg       1.00      1.00      1.00     12000
weighted avg       1.00      1.00      1.00     12000


7단계: 최적화된 랜덤 포레스트 모델 저장

개선된 랜덤 포레스트 모델 학습 및 저장 완료!
완료 시간: 2025-07-22 04:53:46
최종 F1 스코어 (CV): 0.9957
저장된 최적 모델 경로: /content/drive/MyDrive/1차모델저장/optimized_randomforest_model_20250722_045346.pkl
저장된 모델 크기: 17.49 MB


In [6]:
# 1. Kiwi 형태소 분석기 설치
!pip install kiwipiepy -q

# 2. 필요한 라이브러리 임포트
import pandas as pd
import joblib  # 모델을 불러오기 위해 필요
from kiwipiepy import Kiwi
from sklearn.metrics import classification_report # 성능 평가를 위해 필요
from google.colab import drive
import os

In [7]:
# --- 중요: 자신의 Google Drive 경로에 맞게 수정 ---

# 1. 학습 완료 후 저장한 모델 파일 경로
MODEL_PATH = "/content/drive/MyDrive/1차모델저장/optimized_randomforest_model_20250722_045346.pkl" # <-- 이 부분을 수정해!

# 2. 평가할 첫 번째 테스트 데이터셋 경로
TESTSET_1_PATH = "/content/drive/MyDrive/1차모델_테스트데이터셋.csv" # <-- 이 부분을 수정해!

# 3. 평가할 두 번째 테스트 데이터셋 경로
TESTSET_2_PATH = "/content/drive/MyDrive/시나리오통화테스트셋.csv" # <-- 이 부분을 수정해!

# ------------------------------------------------

print(f"불러올 모델: {MODEL_PATH}")
print(f"테스트셋 1: {TESTSET_1_PATH}")
print(f"테스트셋 2: {TESTSET_2_PATH}")

불러올 모델: /content/drive/MyDrive/1차모델저장/optimized_randomforest_model_20250722_045346.pkl
테스트셋 1: /content/drive/MyDrive/1차모델_테스트데이터셋.csv
테스트셋 2: /content/drive/MyDrive/시나리오통화테스트셋.csv


In [8]:
# 1. 저장된 모델 불러오기
# 모델은 TF-IDF 벡터화와 RandomForest 분류기가 포함된 파이프라인 형태여야 해.
print("모델을 불러오는 중...")
model = joblib.load(MODEL_PATH)
print("모델 로딩 완료!")
print(model) # 모델 구조 확인

# 2. 키위 형태소 분석기 및 전처리 함수 정의
# 학습 때 사용한 전처리 함수와 반드시 동일해야 해
kiwi = Kiwi()
def tokenize_and_filter(text):
    result = kiwi.analyze(text)[0][0]
    tokens = []
    for word, pos, _, _ in result:
        if pos in {"NNG", "NNP", "VV", "VA"}:
            if pos in {"VV", "VA"}:
                word = word + "다"
            tokens.append(word)
    return " ".join(tokens)

모델을 불러오는 중...
모델 로딩 완료!
Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.9, min_df=5, ngram_range=(1, 2),
                                 sublinear_tf=True)),
                ('clf',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])


In [9]:
def evaluate_on_test_set(model_pipeline, file_path):
    """
    주어진 경로의 테스트셋을 불러와 전처리하고,
    모델로 예측한 뒤 성능 리포트를 출력하는 함수
    """
    print("\n" + "="*60)
    print(f"테스트 시작: {os.path.basename(file_path)}")

    # 1. 테스트 데이터 로드
    df_test = pd.read_csv(file_path)
    print(f"- 데이터 크기: {df_test.shape}")

    # 2. 학습과 동일한 전처리 적용
    print("- 텍스트 전처리 중...")
    X_test = df_test["text"].astype(str).apply(tokenize_and_filter)
    y_test = df_test["is_phishing"]

    # 3. 모델 예측
    print("- 모델 예측 수행 중...")
    y_pred = model_pipeline.predict(X_test)

    # 4. 성능 리포트 출력
    print("\n[ 최종 성능 평가 리포트 ]")
    report = classification_report(y_test, y_pred, target_names=['정상대화 (0)', '보이스피싱 (1)'])
    print(report)
    print("="*60)

# 첫 번째 테스트셋으로 평가 실행
evaluate_on_test_set(model, TESTSET_1_PATH)

# 두 번째 테스트셋으로 평가 실행
evaluate_on_test_set(model, TESTSET_2_PATH)


테스트 시작: 1차모델_테스트데이터셋.csv
- 데이터 크기: (1000, 3)
- 텍스트 전처리 중...
- 모델 예측 수행 중...

[ 최종 성능 평가 리포트 ]
              precision    recall  f1-score   support

    정상대화 (0)       0.97      1.00      0.99       500
   보이스피싱 (1)       1.00      0.97      0.98       500

    accuracy                           0.98      1000
   macro avg       0.99      0.98      0.98      1000
weighted avg       0.99      0.98      0.98      1000


테스트 시작: 시나리오통화테스트셋.csv
- 데이터 크기: (20, 3)
- 텍스트 전처리 중...
- 모델 예측 수행 중...

[ 최종 성능 평가 리포트 ]
              precision    recall  f1-score   support

    정상대화 (0)       1.00      0.80      0.89        10
   보이스피싱 (1)       0.83      1.00      0.91        10

    accuracy                           0.90        20
   macro avg       0.92      0.90      0.90        20
weighted avg       0.92      0.90      0.90        20

